# 🔬 Climate Sensitivity Proof-of-Concept (PoC) Simulation
This notebook simulates a 24-hour cycle of a solar-powered BLDC refrigerator to evaluate battery State of Charge (SoC) across different ecosystems.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Core hardware configurations
solar_capacity_wp = 500.0      
battery_capacity_wh = 1800.0   
hours = np.arange(0, 24, 1)
clear_sky_irradiance = np.array([0,0,0,0,0,0, 50,200,450,700,850,950, 1000,950,800,600,350,100, 0,0,0,0,0,0])

def simulate_climate(scenario="Standard"):
    efficiency_loss_temp = 1.0
    irradiance_modifier = 1.0
    load_multiplier = 1.0
    
    if scenario == "Arid Zone (Blistering 45°C)":
        efficiency_loss_temp = 0.82  
        load_multiplier = 1.45       
        irradiance = clear_sky_irradiance
    elif scenario == "Rainforest / Monsoon (Overcast 30°C)":
        efficiency_loss_temp = 0.98  
        irradiance_modifier = 0.35   
        load_multiplier = 1.10       
        irradiance = clear_sky_irradiance * irradiance_modifier
    else:
        efficiency_loss_temp = 0.92  
        irradiance = clear_sky_irradiance

    hourly_gen = (irradiance / 1000.0) * solar_capacity_wp * 0.85 * efficiency_loss_temp
    base_fridge_load = np.array([45,45,45,45,45,45, 60,60,90,90,90,90, 90,90,90,90,110,110, 60,60,45,45,45,45])
    hourly_load = base_fridge_load * load_multiplier
    
    soc_tracker = np.zeros(24)
    current_charge = battery_capacity_wh * 0.70 
    
    for t in hours:
        net_power = hourly_gen[t] - hourly_load[t]
        current_charge += net_power
        if current_charge > battery_capacity_wh: current_charge = battery_capacity_wh
        if current_charge < 0: current_charge = 0
        soc_tracker[t] = (current_charge / battery_capacity_wh) * 100
        
    return hourly_gen, hourly_load, soc_tracker

plt.figure(figsize=(12, 6))
scenarios = ["Standard", "Arid Zone (Blistering 45°C)", "Rainforest / Monsoon (Overcast 30°C)"]
colors = ['g', 'r', 'b']

for idx, sc in enumerate(scenarios):
    gen, load, soc = simulate_climate(sc)
    plt.plot(hours, soc, label=f'{sc}', color=colors[idx], lw=2.5)

plt.title('Climatic Sensitivity Analysis: Battery State of Charge (SoC %)')
plt.xlabel('Hour of the Day')
plt.ylabel('Battery %')
plt.axhline(20, color='darkred', linestyle=':')
plt.grid(True, linestyle=':')
plt.legend()
plt.show()